In [1]:
!pip install -q \
langchain \
langchain-community \
langchain-google-genai \
langchain-text-splitters \
langchain-chroma \
chromadb \
wikipedia-api

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.

In [1]:
import langchain
import chromadb
import wikipediaapi

print("All imports successful")

All imports successful


In [2]:
!pip list | grep chroma

chromadb                                 1.5.9
langchain-chroma                         1.1.0


In [3]:
import wikipediaapi
import os

wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='rag-internship-project'
)

topics = [
    "Artificial intelligence",
    "Machine learning",
    "Deep learning",
    "Neural network",
    "Large language model",
    "Generative artificial intelligence",
    "Natural language processing",
    "Computer vision",
    "Reinforcement learning",
    "Retrieval-augmented generation"
]

os.makedirs("wiki_docs", exist_ok=True)

for topic in topics:
    page = wiki.page(topic)

    if page.exists():
        filename = topic.replace(" ", "_") + ".txt"

        with open(
            f"wiki_docs/{filename}",
            "w",
            encoding="utf-8"
        ) as f:
            f.write(page.text)

        print(f"Saved: {filename}")

    else:
        print(f"Page not found: {topic}")

Saved: Artificial_intelligence.txt
Saved: Machine_learning.txt
Saved: Deep_learning.txt
Saved: Neural_network.txt
Saved: Large_language_model.txt
Saved: Generative_artificial_intelligence.txt
Saved: Natural_language_processing.txt
Saved: Computer_vision.txt
Saved: Reinforcement_learning.txt
Saved: Retrieval-augmented_generation.txt


Verify Files

In [4]:
import os

files = os.listdir("wiki_docs")

print("Number of documents:", len(files))

for file in files:
    print(file)

Number of documents: 10
Computer_vision.txt
Natural_language_processing.txt
Generative_artificial_intelligence.txt
Deep_learning.txt
Retrieval-augmented_generation.txt
Large_language_model.txt
Machine_learning.txt
Neural_network.txt
Artificial_intelligence.txt
Reinforcement_learning.txt


**Ingestion** **Pipeline**

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

loader = DirectoryLoader(
    "wiki_docs",
    glob="*.txt",
    loader_cls=TextLoader
)

documents = loader.load()

print("Documents loaded:", len(documents))

/tmp/ipykernel_870/2037484324.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader


Documents loaded: 10


check loaded data

In [6]:
print(documents[0].page_content[:1000])

Computer vision tasks include methods for acquiring, processing, analyzing, and understanding digital images, and extraction of high-dimensional data from the real world in order to produce numerical or symbolic information, e.g. in the form of decisions. "Understanding" in this context signifies the transformation of visual images into descriptions of the world that make sense to thought processes and can elicit appropriate action. This image understanding can be seen as the disentangling of symbolic information from image data using models constructed with the aid of geometry, physics, statistics, and learning theory.
The scientific discipline of computer vision is concerned with the theory behind artificial systems that extract information from images. Image data can take many forms, such as video sequences, views from multiple cameras, multi-dimensional data from a 3D scanner, 3D point clouds from LiDaR sensors, or medical scanning devices. The technological discipline of computer 

Download documents

In [25]:
!zip -r wiki_docs.zip wiki_docs

  adding: wiki_docs/ (stored 0%)
  adding: wiki_docs/Computer_vision.txt (deflated 65%)
  adding: wiki_docs/Natural_language_processing.txt (deflated 64%)
  adding: wiki_docs/Generative_artificial_intelligence.txt (deflated 61%)
  adding: wiki_docs/Deep_learning.txt (deflated 64%)
  adding: wiki_docs/Retrieval-augmented_generation.txt (deflated 60%)
  adding: wiki_docs/Large_language_model.txt (deflated 63%)
  adding: wiki_docs/Machine_learning.txt (deflated 65%)
  adding: wiki_docs/Neural_network.txt (deflated 59%)
  adding: wiki_docs/Artificial_intelligence.txt (deflated 63%)
  adding: wiki_docs/Reinforcement_learning.txt (deflated 72%)


In [26]:
from google.colab import files

files.download("wiki_docs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Chunk Documents

In [7]:
import langchain
print(langchain.__version__)

1.3.4


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=2500,
    chunk_overlap=250
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 244


In [9]:
print(chunks[0].page_content)

Computer vision tasks include methods for acquiring, processing, analyzing, and understanding digital images, and extraction of high-dimensional data from the real world in order to produce numerical or symbolic information, e.g. in the form of decisions. "Understanding" in this context signifies the transformation of visual images into descriptions of the world that make sense to thought processes and can elicit appropriate action. This image understanding can be seen as the disentangling of symbolic information from image data using models constructed with the aid of geometry, physics, statistics, and learning theory.
The scientific discipline of computer vision is concerned with the theory behind artificial systems that extract information from images. Image data can take many forms, such as video sequences, views from multiple cameras, multi-dimensional data from a 3D scanner, 3D point clouds from LiDaR sensors, or medical scanning devices. The technological discipline of computer 

In [10]:
print("Documents:", len(documents))
print("Chunks:", len(chunks))
print("Average chunks per document:", len(chunks)/len(documents))

Documents: 10
Chunks: 244
Average chunks per document: 24.4


**Embedding Model**

In [11]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get(
    "Gemini_key"
)

In [12]:
import google.generativeai as genai
from google.colab import userdata

genai.configure(
    api_key=userdata.get("Gemini_key")
)

for model in genai.list_models():
    if "embed" in model.name.lower():
        print(model.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [13]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [14]:
from langchain_chroma import Chroma

db = Chroma(
    collection_name="test",
    embedding_function=embeddings,
    persist_directory="chroma_db"
)

print("Chroma initialized successfully")

Chroma initialized successfully


Store in ChromaDB

In [53]:
import chromadb
import langchain

print("chromadb:", chromadb.__version__)
print("langchain:", langchain.__version__)

chromadb: 1.5.9
langchain: 1.3.4


In [16]:
from langchain_chroma import Chroma
import time

persist_dir = "chroma_db"

vectorstore = None
batch_size = 20

for i in range(0, len(chunks), batch_size):

    batch = chunks[i:i+batch_size]

    print(f"Processing chunks {i+1} to {min(i+batch_size, len(chunks))}")

    if vectorstore is None:
        vectorstore = Chroma.from_documents(
            documents=batch,
            embedding=embeddings,
            persist_directory=persist_dir,
            collection_name="wiki_rag"
        )
    else:
        vectorstore.add_documents(batch)

    print("✓ Batch stored successfully")

    time.sleep(35)

print("✓ All chunks stored in ChromaDB")

Processing chunks 1 to 20
✓ Batch stored successfully
Processing chunks 21 to 40
✓ Batch stored successfully
Processing chunks 41 to 60
✓ Batch stored successfully
Processing chunks 61 to 80
✓ Batch stored successfully
Processing chunks 81 to 100
✓ Batch stored successfully
Processing chunks 101 to 120
✓ Batch stored successfully
Processing chunks 121 to 140
✓ Batch stored successfully
Processing chunks 141 to 160
✓ Batch stored successfully
Processing chunks 161 to 180
✓ Batch stored successfully
Processing chunks 181 to 200
✓ Batch stored successfully
Processing chunks 201 to 220
✓ Batch stored successfully
Processing chunks 221 to 240
✓ Batch stored successfully
Processing chunks 241 to 244
✓ Batch stored successfully
✓ All chunks stored in ChromaDB


In [17]:
print(vectorstore._collection.count())

244
